# Proyecto Final — ¿Qué detector despliegas, y por qué?
Pipeline completo: YOLOv8n (cerrado) vs YOLO-World (vocab. abierto) vs Florence-2 + Moondream2 (VLM).

**Antes de correr:** activar GPU T4 (Entorno de ejecución > Cambiar tipo > GPU > T4).

## 0. Setup: clonar repo, instalar dependencias, configurar API keys

In [ ]:
# --- OPCION A: ya subiste el repo a tu propio GitHub ---
# Reemplaza la URL de abajo por la de TU repo (con tu usuario real)
REPO_URL = "https://github.com/TU_USUARIO/TU_REPO.git"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

import os
if not os.path.isdir(REPO_NAME):
    !git clone {REPO_URL}
%cd {REPO_NAME}
!pip install -q -r requirements.txt

# --- OPCION B: no quieres usar GitHub, solo subir el .zip a Colab ---
# 1) Sube proyecto_final_repo.zip con el icono de carpeta (panel izquierdo) a /content
# 2) Comenta el bloque de arriba (Opcion A) y descomenta estas 3 lineas:
# import zipfile, os
# zipfile.ZipFile("/content/proyecto_final_repo.zip").extractall("/content/proyecto")
# %cd /content/proyecto/repo
# !pip install -q -r requirements.txt


> **Importante:** si al instalar `requirements.txt` Colab muestra un banner
> rojo diciendo que reinicies el entorno de ejecución (por conflicto de
> versión de `numpy` u otra librería preinstalada), hazlo:
> `Entorno de ejecución > Reiniciar sesión`, y luego sigue ejecutando
> las celdas desde la Celda 3 en adelante (no hace falta repetir la instalación).

In [ ]:
import os
os.environ["ROBOFLOW_API_KEY"] = "TU_API_KEY_AQUI"  # ver Guia_Cuentas_y_Setup.md

from huggingface_hub import login
# login("TU_HF_TOKEN_AQUI")  # opcional, evita limites de descarga

In [ ]:
import sys
sys.path.append(os.path.abspath("."))

from src.config import *
from src.utils.reproducibility import set_global_seed
set_global_seed(SEED)

## 1. Descargar CSS-Data

In [ ]:
from src.data.download import download_css_data
dataset_dir = download_css_data()
data_yaml_path = os.path.join(dataset_dir, "data.yaml")

## 2. Cargar frames del split test (feed de monitoreo compartido)

In [ ]:
from src.data.frame_sampler import load_test_frames
frames = load_test_frames(dataset_dir)
image_paths = [f.image_path for f in frames]
print(f"Total de frames de test: {len(image_paths)}")

### Ground truth por frame (para todas las métricas)
Se parsean los labels YOLO (.txt) a clases por frame, para nivel-caja y nivel-evento.

In [ ]:
def parse_yolo_label(label_path, class_names, img_w, img_h):
    boxes, classes = [], []
    if label_path is None or not os.path.exists(label_path):
        return boxes, classes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id, xc, yc, w, h = int(parts[0]), *map(float, parts[1:5])
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append([x1, y1, x2, y2])
            classes.append(class_names[cls_id])
    return boxes, classes

from PIL import Image
all_gts = []
for f in frames:
    with Image.open(f.image_path) as im:
        w, h = im.size
    boxes, classes = parse_yolo_label(f.label_path, CSS_CLASSES, w, h)
    all_gts.append({"boxes": boxes, "classes": classes})

## 3. Paradigma 1 — YOLOv8n (cerrado): fine-tuning + inferencia

In [ ]:
from src.models.yolo_closed import train_yolo_closed, predict_yolo_closed

_, best_ckpt = train_yolo_closed(data_yaml_path)
preds_yolo_closed = predict_yolo_closed(best_ckpt, image_paths)

## 4. Paradigma 2 — YOLO-World (vocabulario abierto)

In [ ]:
from src.models.yolo_world import load_yolo_world, predict_yolo_world

yolo_world_model = load_yolo_world()
preds_yolo_world = predict_yolo_world(yolo_world_model, image_paths)

## 5. Paradigma 3 — Florence-2 (cajas) + Moondream2 (pregunta-evento)

In [ ]:
from src.models.florence2 import load_florence2, predict_florence2

florence_model, florence_processor = load_florence2()
preds_florence2 = predict_florence2(florence_model, florence_processor, image_paths)

In [ ]:
from src.models.moondream import load_moondream, predict_moondream, run_prompt_robustness

moondream_model, moondream_tokenizer = load_moondream()
main_prompt = MOONDREAM_PROMPT_VARIANTS[0]
preds_moondream = predict_moondream(moondream_model, moondream_tokenizer, image_paths, main_prompt)

## 6. Métricas — (a) Precisión: mAP/IoU (nivel-caja) y F1 (nivel-evento)

In [ ]:
from src.metrics.box_metrics import compute_map50

map_yolo_closed, ap_per_class_closed = compute_map50(preds_yolo_closed, all_gts, SHARED_BOX_CLASSES)
map_yolo_world, ap_per_class_world = compute_map50(preds_yolo_world, all_gts, SHARED_BOX_CLASSES)
map_florence2, ap_per_class_florence = compute_map50(preds_florence2, all_gts, SHARED_BOX_CLASSES)

print("mAP@0.5 YOLOv8n:", map_yolo_closed)
print("mAP@0.5 YOLO-World:", map_yolo_world)
print("mAP@0.5 Florence-2:", map_florence2)

In [ ]:
from src.metrics.event_metrics import (
    frame_has_violation_gt, frame_has_violation_pred_boxes, compute_event_level_f1
)

y_true = [frame_has_violation_gt(gt["classes"], VIOLATION_CLASS) for gt in all_gts]

y_pred_closed = [frame_has_violation_pred_boxes(p["classes"], VIOLATION_CLASS) for p in preds_yolo_closed]
y_pred_world = [frame_has_violation_pred_boxes(p["classes"], VIOLATION_CLASS) for p in preds_yolo_world]
y_pred_florence = [frame_has_violation_pred_boxes(p["classes"], VIOLATION_CLASS) for p in preds_florence2]
y_pred_moondream = [p["violation_predicted"] for p in preds_moondream]

f1_closed = compute_event_level_f1(y_true, y_pred_closed)
f1_world = compute_event_level_f1(y_true, y_pred_world)
f1_florence = compute_event_level_f1(y_true, y_pred_florence)
f1_moondream = compute_event_level_f1(y_true, y_pred_moondream)

print("F1 nivel-evento — YOLOv8n:", f1_closed)
print("F1 nivel-evento — YOLO-World:", f1_world)
print("F1 nivel-evento — Florence-2:", f1_florence)
print("F1 nivel-evento — Moondream2:", f1_moondream)

## 7. Métricas — (b) Velocidad: FPS / latencia

In [ ]:
from src.metrics.speed_metrics import compute_speed_stats

speed_closed = compute_speed_stats([p["latency_s"] for p in preds_yolo_closed])
speed_world = compute_speed_stats([p["latency_s"] for p in preds_yolo_world])
speed_florence = compute_speed_stats([p["latency_s"] for p in preds_florence2])
speed_moondream = compute_speed_stats([p["latency_s"] for p in preds_moondream])

print("FPS YOLOv8n:", speed_closed["fps"])
print("FPS YOLO-World:", speed_world["fps"])
print("FPS Florence-2:", speed_florence["fps"])
print("FPS Moondream2:", speed_moondream["fps"])

## 8. Métricas — (c) Robustez: sensibilidad al prompt (VLM) y clase no vista (YOLO)

In [ ]:
from src.metrics.robustness import compute_prompt_sensitivity, compute_unseen_class_performance

sample_paths = image_paths[:50]
sample_y_true = y_true[:50]

all_prompt_results = run_prompt_robustness(
    moondream_model, moondream_tokenizer, sample_paths, MOONDREAM_PROMPT_VARIANTS
)

f1_per_prompt = {}
for prompt, results in all_prompt_results.items():
    y_pred_prompt = [r["violation_predicted"] for r in results]
    f1_per_prompt[prompt] = compute_event_level_f1(sample_y_true, y_pred_prompt)["f1"]

prompt_sensitivity = compute_prompt_sensitivity(f1_per_prompt)
print("Sensibilidad al prompt:", prompt_sensitivity)

unseen_result = compute_unseen_class_performance(preds_yolo_closed, unseen_class="gloves")
print("Desempeño en clase no vista (YOLOv8n):", unseen_result)

## 9. Tablas y gráficos comparativos (guardados en results/)

In [ ]:
from src.utils.visualization import (
    save_comparison_table, plot_bar_comparison, plot_confusion_matrix, plot_prompt_sensitivity
)

comparison = {
    "YOLOv8n (cerrado)": {
        "mAP@0.5": map_yolo_closed, "F1_evento": f1_closed["f1"],
        "FPS": speed_closed["fps"], "licencia": "AGPL-3.0",
    },
    "YOLO-World (vocab. abierto)": {
        "mAP@0.5": map_yolo_world, "F1_evento": f1_world["f1"],
        "FPS": speed_world["fps"], "licencia": "GPL-3.0",
    },
    "Florence-2 (VLM grounding)": {
        "mAP@0.5": map_florence2, "F1_evento": f1_florence["f1"],
        "FPS": speed_florence["fps"], "licencia": "MIT",
    },
    "Moondream2 (VLM generativo)": {
        "mAP@0.5": None, "F1_evento": f1_moondream["f1"],
        "FPS": speed_moondream["fps"], "licencia": "Apache-2.0",
    },
}

save_comparison_table(comparison, filename="comparacion_final")
plot_bar_comparison(comparison, "F1_evento", "F1 nivel-evento por paradigma", "F1", "f1_evento_comparacion")
plot_bar_comparison(comparison, "FPS", "FPS por paradigma (misma T4)", "FPS", "fps_comparacion")
plot_confusion_matrix(f1_moondream["confusion_matrix"], ["No violación", "Violación"],
                       "Matriz de confusión — Moondream2", "cm_moondream")
plot_prompt_sensitivity(prompt_sensitivity)

## 10. Conclusión
Ver `README.md` para el informe breve, el cuadro comparativo final y la
respuesta a la pregunta: **¿cuál de los tres sistemas desplegarían, y por qué?**